In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Utilizarea volumului de lucru

<span id="usage"></span>
Utilizarea reprezintă consumul serviciului Qiskit Runtime și este determinată de timpul cât un QPU este blocat pentru executarea volumelor de lucru.

- Utilizarea Session este măsurată ca timpul scurs cât sesiunea rămâne activă, deoarece capacitatea QPU este rezervată pe durata sesiunii, indiferent dacă volumele de lucru rulează activ. Consultă [Lungimea sesiunii](/guides/run-jobs-session#session-length) pentru mai multe informații despre tranzițiile stării sesiunii.
- Utilizarea Batch este măsurată ca timpul cumulativ cât QPU este blocat pentru executarea tuturor joburilor din lot.
- Utilizarea unui job individual este măsurată ca timpul cât QPU este blocat pentru executarea jobului.

Reține că joburile eșuate sau anulate contează în utilizarea ta în anumite circumstanțe - consultă secțiunea [Joburi eșuate și anulate](#failed-job) pentru detalii.

Pentru utilizatorii cu Planul Pay-As-You-Go, consultă [Gestionarea costurilor](/guides/manage-cost) pentru detalii despre setarea unei limite de cost.

<span id="failed-job"></span>
## Utilizarea pentru joburi eșuate și anulate
Când un job eșuează sau este anulat, utilizarea raportată este după cum urmează:

* Modul job sau batch: Dacă eșecul sau anularea a survenit din cauza unei erori de sistem, utilizarea raportată este zero. Pentru joburile care au eșuat din cauza unei erori de utilizator sau când un utilizator a anulat un job, utilizarea raportată este orice consum care a avut loc până în acel moment.

* Modul Session: Utilizarea raportată este timpul de ceas al peretelui cât sesiunea este activă, indiferent de numărul de joburi care eșuează sau sunt anulate.

<span id="view-usage"></span>
## Interogarea utilizării reale a unui volum de lucru
După ce un volum de lucru s-a finalizat, există mai multe modalități de a vizualiza utilizarea reală:

- Rulează [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) sau [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) în `qiskit-ibm-runtime` 0.30 sau ulterior.  Dacă folosești o versiune mai veche de `qiskit-ibm-runtime` (>= 0.23 și < 0.30), utilizarea poate fi găsită în continuare în `session.details()["usage_time"]` și `batch.details()["usage_time"]`.
- Folosește [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) pentru a vedea utilizarea pentru un anumit lot sau sesiune.
- Folosește [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) pentru a vedea utilizarea pentru un job individual.

<span id="instance-usage"></span>
## Vizualizarea utilizării instanței
Poți vizualiza utilizarea unei instanțe pe pagina [Instanțe](https://quantum.cloud.ibm.com/instances) sau, pentru cei cu autoritatea corespunzătoare, pe pagina [Analiză](https://quantum.cloud.ibm.com/analytics).  Reține că paginile pot afișa numere de utilizare diferite deoarece calculează utilizarea în mod diferit.

Pagina Instanțe afișează utilizarea în timp real pentru ultimele 28 de zile (rulant), până la ora curentă din ziua curentă.  Utilizarea de pe pagina Analiză este recalculată din oră în oră și include ultimele 28 de zile complete; adică, afișează utilizarea de la 00:00 acum 28 de zile până astăzi, la ora rotundă.

## Estimarea utilizării înainte de a trimite un job
Deși obținerea unei estimări locale precise este complicată de operațiunile suplimentare efectuate pentru suprimarea și atenuarea erorilor, poți folosi această formulă de bază pentru a obține o aproximare a utilizării estimate:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` este un overhead de aproximativ 2s per sub-job. Aceasta include operațiuni precum încărcarea datelor utile în electronica de control. Jobul tău primitiv poate fi împărțit în mai multe sub-joburi dacă este prea mare pentru ca motorul de execuție să îl proceseze dintr-o dată.
- `rep_delay` este o opțiune [personalizabilă de utilizator](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay), iar valoarea implicită este dată de `backend.default_rep_delay`, care este de 250 de microsecunde pe majoritatea Backend-urilor IBM Quantum. Reține că reducerea `rep_delay` scade timpul total de execuție al QPU, dar cu costul unei rate crescute de erori la pregătirea stării; consultă ghidul [Execuție cu rată de repetiție dinamică](/guides/repetition-rate-execution) pentru mai multe informații.
- `<circuit length>` este lungimea totală a instrucțiunilor. Fiecare instrucțiune durează un timp diferit pe QPU, deci lungimea totală variază de la circuit la circuit. O măsurătoare, de exemplu, poate dura de 56 de ori mai mult decât o poartă `x`. `backend.target[<instruction>][<qubit>].duration` poate fi folosit pentru a găsi durata exactă pentru fiecare instrucțiune. O lungime tipică a unui circuit este probabil între 50-100 de microsecunde. Dacă folosești tehnici de suprimare sau atenuare a erorilor cu primitivele, instrucțiuni suplimentare ar putea fi inserate în circuitul tău, ceea ce ar crește lungimea totală a circuitul-ului.
    > **Note:** [Opțiunea experimentală `scheduler_timing`](/guides/visualize-circuit-timing) returnează timpul total al circuitului, dar acesta NU este timpul folosit pentru facturare.
- `<num executions>` este numărul total de circuite înmulțit cu numărul de împușcături, unde circuitele sunt cele generate după ce elementele PUB sunt difuzate.
    - Dacă folosești tehnici de atenuare a erorilor cu primitivele, circuite suplimentare ar putea fi rulate ca parte a procesului de atenuare, ceea ce ar crește numărul total de execuții. În plus, tehnicile avansate de atenuare a erorilor, cum ar fi PEA și PEC, vin cu un overhead mult mai mare deoarece necesită rularea circuitelor pentru învățarea zgomotului.
    - Estimator grupează observabilele care comutează bit cu bit, ceea ce reduce numărul de execuții.

Dacă nu folosești tehnici avansate de atenuare a erorilor sau `rep_delay` personalizat, poți folosi `2+0.00035*<num executions>` ca formulă rapidă.

### Estimează utilizarea local cu Qiskit
Acest exemplu de cod demonstrează cum să folosești Qiskit pentru a calcula timpul circuitului: